# Assistente de Investimentos (B3) - Demonstracao

AI Agent multi-agente (Google ADK) que recomenda **COMPRAR / VENDER / AGUARDAR** para `VALE3`, `PETR4`, `BBAS3` e `ITUB4`, combinando indicadores tecnicos e sentimento de noticias (FinBERT-PT-BR), com raciocinio explicavel (ReAct / Chain-of-Thought).

Este notebook demonstra: (1) as tools de coleta/processamento, (2) a recomendacao consolidada por acao, (3) o backtest vs Buy-and-Hold e (4) o agente LLM completo.

> Execute com o kernel do ambiente Poetry: `poetry run python -m ipykernel install --user --name assistente` e selecione-o, ou rode `poetry run jupyter lab`.

## 1. Setup

In [ ]:
import json
import pandas as pd

from assistente.config import TICKERS

pd.set_option('display.max_columns', None)
list(TICKERS.items())

## 2. Tools (Percepcao -> Processamento)

Preco/volume (yfinance), indicadores tecnicos (pandas-ta-classic) e sentimento de noticias (FinBERT-PT-BR).

In [ ]:
from assistente.tools.market_data import get_price_data
from assistente.tools.indicators import calculate_indicators

print(json.dumps(get_price_data('VALE3', '3mo'), ensure_ascii=False, indent=2)[:500])
print(json.dumps(calculate_indicators('VALE3', '6mo'), ensure_ascii=False, indent=2))

In [ ]:
from assistente.tools.sentiment import analyze_news_sentiment

sent = analyze_news_sentiment('PETR4')
print('sentiment_available:', sent.get('sentiment_available'), '| score:', sent.get('aggregate_sentiment'))
for h in sent.get('headlines', [])[:5]:
    print('-', h.get('sentiment', '?'), '|', h['title'][:90])

## 3. Recomendacao consolidada (baseline deterministico)

Combina indicadores + sentimento em um score explicavel por acao, no formato do descritivo.

In [ ]:
from assistente.tools.recommendation import generate_recommendation

rows = [generate_recommendation(t, '6mo') for t in TICKERS]
df = pd.DataFrame(rows)[
    ['ticker', 'last_close', 'rsi', 'macd_signal', 'trend', 'news_sentiment', 'weighted_score', 'recommendation']
]
df

## 4. Backtest vs Buy-and-Hold

Acuracia (acerto de tendencia em `horizon` dias) e retorno simulado da estrategia comparado a comprar e segurar. Usa o motor deterministico (causal, sem lookahead); o sentimento historico nao esta disponivel, logo o backtest e tecnico.

In [ ]:
from assistente.backtest import run_backtest_all

bt = run_backtest_all(list(TICKERS), period='2y', horizon=5)
bt[['ticker', 'n_signals', 'accuracy', 'strategy_return_pct', 'buy_hold_return_pct', 'outperformance_pct']]

In [ ]:
from assistente.tools.charting import plot_chart
from IPython.display import Image

res = plot_chart('VALE3', '6mo')
Image(res['image_path']) if res['status'] == 'success' else res

## 5. Agente LLM completo (ReAct + Chain-of-Thought)

Requer `ANTHROPIC_API_KEY` no `.env` (ou troque `ADK_MODEL` para um modelo Ollama local). O orquestrador consulta o agente tecnico e o de noticias e decide com o raciocinio explicado.

In [ ]:
from assistente.runner import AgentSession

session = AgentSession()
resposta = await session.ask('Analise a VALE3 e recomende uma acao, explicando o raciocinio.')
print(resposta)